# Projeto: Funções Para a Geração do Boletim Climático

## Descrição Geral
Este notebook contém as funções necessárias para gerar um boletim climático e posteriormente encaminhar para uma lista de emails.

## Objetivos

- Centralizar funções necessárias para a geração do boletim climático.
- Facilitar o processo de manutenção.
- Documentação.


## Tecnologias Utilizadas

- Apache Spark (PySpark)
- Databricks Notebooks
- Delta Lake
- CWI Rocket
- SMTP do Gmail

---

## Funções Definidas

| Função                     | Descrição                                                                 |
|---------------------------|---------------------------------------------------------------------------|
| `enviar_email`            | Faz o envio de emails para uma lista determinada |
| `chat_rocket`            | Faz interações com o rocket via api por meio dos workspaces |
| `converter_para_rocket`            | Transforma um dataframe para texto |

---

### Histórico de Atualizações

| Data       | Responsável  | Versão | Descrição                                      |
|------------|--------------|--------|-----------------------------------------------|
| 25/05/2025 | Gabriel Rost  | 1.0    | Configuração de Imports e Documentação     |






In [0]:
%run ../../config/enum_config

In [0]:
CHAVE_API_ROCKET = enum_configs.CHAVE_API_ROCKET.value
SENHA_APP_GMAIL = enum_configs.SENHA_APP_GMAIL.value
EMAIL_REMETENTE = enum_configs.EMAIL_REMETENTE.value

In [0]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

def enviar_email(destinatarios, assunto, corpo):
    """
    Envia um e-mail HTML para uma ou mais pessoas via SMTP do Gmail.

    Parâmetros:
    -----------
    destinatarios : str | list
        Um endereço de e-mail ou uma lista de endereços.

    assunto : str
        Assunto do e-mail.

    corpo : str
        Conteúdo HTML do e-mail.
    """
    email_remetente = EMAIL_REMETENTE
    senha_app = SENHA_APP_GMAIL

    if isinstance(destinatarios, str):
        destinatarios = [destinatarios]

    # Monta o e-mail
    msg = MIMEMultipart("alternative")
    msg['Subject'] = assunto
    msg['From'] = email_remetente
    #msg['To'] = ", ".join(destinatarios)
    
    # Anexa o conteúdo HTML
    parte_html = MIMEText(corpo, "html")
    msg.attach(parte_html)

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(email_remetente, senha_app)
            smtp.sendmail(email_remetente, destinatarios, msg.as_string())
            print("E-mail enviado com sucesso para:", ", ".join(destinatarios))
    except Exception as e:
        print(f"Erro ao enviar e-mail: {e}")

In [0]:
import requests
import json

def chat_rocket(workspace, message):
    """
    Envia uma mensagem para a API de chat da Rocket (CWI) e retorna a resposta do modelo.

    Parâmetros:
    -----------
    workspace : str
        Identificador do workspace (espaço de trabalho) na plataforma da CWI.
    
    message : str
        Mensagem de entrada que será enviada ao modelo de chat.

    Retorno:
    --------
    str
        Resposta em texto fornecida pelo modelo da Rocket, extraída do campo "textResponse".
    """
    TOKEN_BEARER = CHAVE_API_ROCKET
    WORKSPACE = workspace
    MESSAGE = message
    URL_BASE = f"https://chat-api.cwi.com.br/api/v1/workspace/{WORKSPACE}/chat"

    headers = {
        "Authorization": f"Bearer {TOKEN_BEARER}",
        "Content-Type": "application/json"
    }
    data = {
        "message": MESSAGE,
        "mode": "chat"
    }

    response = requests.post(URL_BASE, headers=headers, json=data)
    resposta = response.json().get("textResponse")
    return resposta

In [0]:
def converter_para_rocket(df):
    """
    Converte os dados de um DataFrame Spark para uma string formatada.

    Esta função coleta todos os dados do DataFrame distribuído usando o método `collect()`,
    e os transforma em uma string formatada, útil para envio a APIs ou sistemas de mensagens 
    como o Rocket.

    Atenção: O método `collect()` traz todos os dados para o driver. Isso pode causar 
    problemas de desempenho ou estouro de memória se o DataFrame for muito grande.

    Args:
        df (pyspark.sql.DataFrame): DataFrame Spark contendo os dados que serão convertidos.

    Returns:
        str: Uma string formatada contendo os dados do DataFrame no formato:
             "Dados: [Row(...), Row(...), ...]"
    """
    df = f"""Dados: {df.collect()}"""
    return df